In [25]:
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="descriptives")

### Inflation data
- Import mm23.xlsx from dirs.input_dir, and load sheet 'data'
- The first few rows are filler, so we will need to find the first row where the index is 'CDID' and make that the header
- This is probably row 2, so we should write an assert statement to check that
- Then, filter all columns which are not inflation_cdid = 'L522'
- Rows are either 'filler', 'yearly', 'quarterly' or 'monthly'
- Filter out 'filler' rows. Filter out 'monthly' rows. 'Filter out 'quarterly' rows.
- Filter out 'yearly' rows which are not in the range 2005 - 2026
- Output the remaining series

In [26]:
import pandas as pd

inflation_cdid = 'L522' # CPIH INDEX 0.0
start_year_inf = 2005
xlsx_file = f"{dirs.input_dir}/mm23.xlsx"
xlsx_sheet = "data"

df_raw = pd.read_excel(xlsx_file, sheet_name=xlsx_sheet, engine="calamine")

# Make the first column the index
df_raw.set_index(df_raw.columns[0], inplace=True)

# Make the first row the header
assert df_raw.index[0] == 'CDID', "Expected header not found"
df_raw.columns = df_raw.iloc[0]
df = df_raw[1:]

# Filter all columns where the column name is not our inflation_cdid variable
df = df[df.columns[df.columns == inflation_cdid]]

# Autodetect which index references are 'filler', 'yearly', 'quarterly' or 'monthly', 
# where filler rows don't contain a numeric value within it
filler_rows = df[~df.index.str.contains(r'\d')].index
yearly_rows = df[df.index.str.contains(r'^\d{4}$')].index
quarterly_rows = df[df.index.str.contains(r'^\d{4}Q\d$')].index
monthly_rows = df[df.index.str.contains(r'^\d{4}M\d{2}$')].index

df_yearly_only = df.loc[yearly_rows]
df_yearly_only.index = df_yearly_only.index.astype(int)

# Filter out 'yearly' rows which are not in the range 2005 -
df_year_range = df_yearly_only[(df_yearly_only.index >= start_year_inf)]

# Find the last year in this range which is not a null / NaN
end_year_inf = df_year_range[df_year_range[inflation_cdid].notnull()].index[-1]
df_year_range_filtered = df_year_range[df_year_range.index <= end_year_inf]
# Find where the value is 100
base_year_inf = df_year_range_filtered[df_year_range_filtered[inflation_cdid] == 100].index[0]
if base_year_inf == None:
    raise ValueError("No base year found where the inflation index is 100.")

print(f"Base year: {base_year_inf}, Start year: {start_year_inf}, End year: {end_year_inf}")

df_inflation = df_year_range_filtered[inflation_cdid].astype(float)
print(df_inflation)

Base year: 2015, Start year: 2005, End year: 2025
Title
2005     79.4
2006     81.4
2007     83.3
2008     86.2
2009     87.9
2010     90.1
2011     93.6
2012     96.0
2013     98.2
2014     99.6
2015    100.0
2016    101.0
2017    103.6
2018    106.0
2019    107.8
2020    108.9
2021    111.6
2022    120.5
2023    128.6
2024    132.9
2025    138.0
Name: L522, dtype: float64


### Deflator index

In [27]:
# Write a full block of code to import GDP deflator from input_dir / quarterlynationalaccountsdatatables.xlsx
# Use a similar coding style (i.e. setting variables first, naming pattern)
# Sheet name is 'yearly_variables'
# This is already loaded as panel data. First column is years but also contains quarters
# Filter out quarters, and then grab the 'YBGB' column, which corresponds to the GDP deflator.
# print resulting dataframe

import pandas as pd

deflator_cdid = 'YBGB' # GDP deflator
start_year_def = 2005

xlsx_file = f"{dirs.input_dir}/quarterlynationalaccountsdatatables.xlsx"
xlsx_sheet = "yearly_variables"
df_raw_def = pd.read_excel(xlsx_file, sheet_name=xlsx_sheet, engine="calamine")

df_raw_def.set_index(df_raw_def.columns[0], inplace=True)
df_raw_def.index = df_raw_def.index.astype(str)
regex_pattern = r'^\d{4}(?:\.0)?$'
yearly_rows_def = df_raw_def[df_raw_def.index.str.contains(regex_pattern, na=False)].index

# (Optional) If it caught the ".0", clean it up so the index is purely the year:
df_yearly_only_def = df_raw_def.loc[yearly_rows_def]
df_yearly_only_def.index = df_yearly_only_def.index.str.replace('.0', '', regex=False).astype(int)

# Parse end year
df_year_range_def = df_yearly_only_def[(df_yearly_only_def.index >= start_year_def)]
end_year_def = df_year_range_def[df_year_range_def[deflator_cdid].notnull()].index[-1]
df_year_range_filtered_def = df_year_range_def[df_year_range_def.index <= end_year_def]

# Parse base year
base_year_def = df_year_range_filtered_def[df_year_range_filtered_def[deflator_cdid] == 100].index[0]
if base_year_def == None:
    raise ValueError("No base year found where the GDP deflator is 100.")

print(f"Base year: {base_year_def}, Start year: {start_year_def}, End year: {end_year_def}")

df_deflator = df_year_range_filtered_def[deflator_cdid].astype(float)

print(df_deflator)

Base year: 2023, Start year: 2005, End year: 2025
Unnamed: 0
2005     64.0
2006     65.9
2007     67.3
2008     69.5
2009     70.9
2010     71.8
2011     73.6
2012     74.7
2013     76.3
2014     77.6
2015     78.1
2016     79.4
2017     80.7
2018     82.2
2019     84.3
2020     88.4
2021     89.0
2022     94.0
2023    100.0
2024    103.9
2025    107.7
Name: YBGB, dtype: float64


In [ ]:
# Combine the two dataframes for columns to be 'cpih' and 'gdpdef'. Index is years
# set a base_year variable to be 2024
# normalise both columns to 2024

base_year = 2024

df_prices = pd.DataFrame({
    'cpih': df_inflation,
    'gdpdef': df_deflator
})
rebase_dict = {
    'cpih': df_prices.loc[base_year, 'cpih'] / df_prices.loc[base_year_inf, 'cpih'],
    'gdpdef': df_prices.loc[base_year, 'gdpdef'] / df_prices.loc[base_year_def, 'gdpdef']
}
df_prices_rebased = df_prices / pd.Series(rebase_dict)

for col in df_prices_rebased.columns:
    if round(df_prices_rebased.loc[base_year, col], 2) == 100:
        continue
    raise ValueError(f"Expected 100 in {base_year} for {col}, got {df_prices_rebased.loc[base_year, col]}")